# 📓 Update dataset with last boxscores and matches

# Please first run full pipeline until merge_clean_seasons_boxscores to have base data to work with and merge

In [1]:

import os
import pandas as pd
from datetime import datetime

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Could not locate src/ directory")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from src.config import *
from src.utils import *
from src.nba_scrapping import *



In [2]:
# ⚙️ Initialisation du run
run_timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
current_season = '2025-26'  # À rendre dynamique si besoin plus tard


In [3]:

# 📁 Préparation des dossiers de sortie
os.makedirs(DATA_LAST_GAMES_DIR, exist_ok=True)
os.makedirs(DATA_LAST_BOXSCORES_BATCHES_DIR, exist_ok=True)

# 🔄 Chargement des historiques si existants
hist_games_path = get_latest_file(DATA_LAST_GAMES_MERGED_DIR)


In [4]:

# 📥 1. Téléchargement des matchs de la saison actuelle
print("\n📥 Téléchargement des nouveaux matchs pour la saison:", current_season)
matchs_output_dir = os.path.join(DATA_LAST_GAMES_DIR, current_season)
last_games_path = download_games_for_seasons([current_season], matchs_output_dir, run_timestamp, max_retries=10)


📥 Téléchargement des nouveaux matchs pour la saison: 2025-26
Extraction saison 2025-26


In [5]:

# 📊 2. Comparaison avec les données historiques pour trouver les nouveaux matchs
games_to_scrape = get_new_games(hist_games_path, last_games_path)
print(f"✅ {len(games_to_scrape)} nouveaux matchs trouvés à scraper.")

if games_to_scrape.empty:
    print("✅ Aucun nouveau match à scraper. Fin du script.")
    exit(0)

218 nouveaux matchs à traiter
✅ 218 nouveaux matchs trouvés à scraper.


In [6]:
games_to_scrape

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,PTS,...,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PLUS_MINUS,SEASON
0,22025,1610612747,LAL,Los Angeles Lakers,0022500002,2025-10-21,LAL vs. GSW,L,240,109,...,7,32,39,23,7,2,19,21,-10.0,2025-26
1,22025,1610612745,HOU,Houston Rockets,0022500001,2025-10-21,HOU @ OKC,L,292,124,...,16,36,52,23,6,5,22,26,-1.0,2025-26
2,22025,1610612760,OKC,Oklahoma City Thunder,0022500001,2025-10-21,OKC vs. HOU,W,290,125,...,11,27,38,29,12,4,11,27,1.0,2025-26
3,22025,1610612744,GSW,Golden State Warriors,0022500002,2025-10-21,GSW @ LAL,W,241,119,...,9,31,40,29,10,4,18,27,10.0,2025-26
4,22025,1610612753,ORL,Orlando Magic,0022500081,2025-10-22,ORL vs. MIA,W,242,125,...,9,37,46,23,8,7,15,20,4.0,2025-26
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,22025,1610612737,ATL,Atlanta Hawks,0022500166,2025-11-04,ATL vs. ORL,W,240,127,...,6,28,34,23,11,6,18,24,15.0,2025-26
214,22025,1610612766,CHA,Charlotte Hornets,0022500168,2025-11-04,CHA @ NOP,L,239,112,...,15,34,49,20,8,6,19,23,-4.0,2025-26
215,22025,1610612753,ORL,Orlando Magic,0022500166,2025-11-04,ORL @ ATL,L,240,112,...,14,27,41,26,9,1,17,30,-15.0,2025-26
216,22025,1610612761,TOR,Toronto Raptors,0022500165,2025-11-04,TOR vs. MIL,W,241,128,...,10,40,50,33,8,6,9,23,28.0,2025-26


In [7]:

# 💾 3. Merge historique + nouveaux matchs
historical_games = pd.read_csv(hist_games_path, low_memory=False, dtype={'GAME_ID': str})
all_games = pd.concat([historical_games, games_to_scrape], ignore_index=True)

games_output_path = save_dataframe_to_csv(all_games, DATA_LAST_GAMES_MERGED_DIR, 'games_merged_all_seasons', run_timestamp)
print(f"✅ Jeux de matchs fusionnés sauvegardés dans {games_output_path}.")


✅ Jeux de matchs fusionnés sauvegardés dans /home/ju/Documents/Dev/NBA_Predictor/data/legacy/raw_last/games_merged/games_merged_all_seasons_2025-11-05_13-35-51.csv.


In [8]:

# 🏀 4. Scraping des nouveaux boxscores
print(f"--- Traitement de la saison {current_season} ---")
season_df = games_to_scrape[games_to_scrape['SEASON'] == current_season]
season_output_dir = os.path.join(DATA_LAST_BOXSCORES_BATCHES_DIR, run_timestamp, current_season)
scrape_boxscores_v3_for_games(season_df, season_output_dir,max_retries=10)

--- Traitement de la saison 2025-26 ---
[DEBUG] Found 0 batch files for endpoint 'traditional' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'advanced' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'fourfactors' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'misc' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'scoring' in season 2025-26
[DEBUG] Found 0 batch files for endpoint 'usage' in season 2025-26
--------- 109 GAME_ID to scrap for season 2025-26 ---------
[1/109] GAME_ID: 0022500002 - 2025-10-21
[2/109] GAME_ID: 0022500001 - 2025-10-21
[3/109] GAME_ID: 0022500081 - 2025-10-22
[4/109] GAME_ID: 0022500088 - 2025-10-22
[5/109] GAME_ID: 0022500083 - 2025-10-22
[6/109] GAME_ID: 0022500089 - 2025-10-22
[7/109] GAME_ID: 0022500082 - 2025-10-22
[8/109] GAME_ID: 0022500080 - 2025-10-22
[9/109] GAME_ID: 0022500004 - 2025-10-22
[10/109] GAME_ID: 0022500086 - 2025-10-22
[11/109] GAME_ID: 0022500085 - 2025-10-22
[12/109] GAME_ID: 0022500084 

True

In [9]:
print("\n✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.")



✅ Mise à jour complète terminée. Données prêtes pour le feature engineering.
